# A2 v4 FP16 continuation
Recovered ~0.5-epoch adapter; train 1.5 additional epochs with fresh optimizer.
This is a warm start, not an exact optimizer/sample-order resume.
T4 uses FP16. Speed guard after 5 steps. No before-evaluation, no reserved test.
Checkpoint labels 0.5/1.0/1.5 refer to this additional stage.


In [ ]:
import json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

WORK = Path('/kaggle/working/a2_v4_fp16_work')
RESULTS = Path('/kaggle/working/results_v4_fp16')
ZIP_OUT = Path('/kaggle/working/a2_sentence_v4_fp16_results.zip')
WORK.mkdir(parents=True, exist_ok=True)

# Locate uploaded Kaggle bundle (zip file OR auto-extracted directory under /kaggle/input)
bundle_zip = next(Path('/kaggle/input').rglob('a2_sentence_v4_fp16_bundle.bundle'), None)
if bundle_zip is None:
    bundle_zip = next(Path('/kaggle/input').rglob('a2_sentence_v4_fp16_bundle.zip'), None)
if bundle_zip is not None:
    print('Extracting bundle:', bundle_zip)
    with zipfile.ZipFile(bundle_zip, 'r') as zf:
        zf.extractall(WORK)
else:
    found_train = next(Path('/kaggle/input').rglob('data/processed/a2_sentence_selection_v4/train.jsonl'), None)
    if found_train is None:
        raise FileNotFoundError('Upload a2_sentence_v4_fp16_bundle.zip as a Kaggle Dataset first.')
    src_root = found_train.parents[3]
    print('Copying extracted dataset from:', src_root)
    for sub in ('data', 'initial_adapter', 'src', 'scripts'):
        dst = WORK / sub
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src_root / sub, dst)

os.chdir(WORK)
sys.path.insert(0, str(WORK))
print('Workspace ready:', WORK)

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORK/'requirements-colab-a2.txt')])
subprocess.check_call(['nvidia-smi'])

In [ ]:
import subprocess, sys
env = dict(os.environ, PYTHONPATH=str(WORK))
subprocess.check_call([
    sys.executable, '-m', 'scripts.train_a2_v4_kaggle',
    '--preflight',
    '--dataset-dir', str(WORK / 'data/processed/a2_sentence_selection_v4'),
    '--initial-adapter', str(WORK / 'initial_adapter'),
    '--expected-adapter-sha256', 'e3b5a36f19bff6df97234b967678a81208039df7908bc4a19883b723fde96a66',
], env=env)

In [ ]:
RESULTS.mkdir(parents=True,exist_ok=True)
(RESULTS/'continuation_plan.json').write_text((WORK/'continuation_plan.json').read_text())
import subprocess, sys
env = dict(
    os.environ,
    PYTHONPATH=str(WORK),
    CUDA_VISIBLE_DEVICES='0',
    USE_TF='0', USE_FLAX='0', PYTHONUNBUFFERED='1',
    TOKENIZERS_PARALLELISM='false',
    PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',
)
subprocess.check_call([
    sys.executable, '-m', 'scripts.train_a2_v4_kaggle',
    '--dataset-dir', str(WORK / 'data/processed/a2_sentence_selection_v4'),
    '--initial-adapter', str(WORK / 'initial_adapter'),
    '--expected-adapter-sha256', 'e3b5a36f19bff6df97234b967678a81208039df7908bc4a19883b723fde96a66',
    '--output-dir', str(RESULTS),
    '--epochs', '1.5',
    '--lr', '8e-5',
    '--batch-size', '1',
    '--grad-accum', '8',
    '--max-length', '4096',
    '--results-zip', str(ZIP_OUT),
], env=env)
print('Saved results zip:', ZIP_OUT, 'exists=', ZIP_OUT.exists())

In [ ]:
summary = json.loads((RESULTS / 'checkpoint_summary.json').read_text(encoding='utf-8'))
print('Selected checkpoint:', summary['selected_checkpoint'], 'score:', round(summary['selected_composite_score'], 4))
for tag, row in summary['checkpoints'].items():
    v, c = row['val'], row['challenge']
    print(
        f"{tag:10s} comp={row['composite_score']:.4f} | "
        f"val(status={v['status_correct']:.3f}, f1={v['selection_f1_on_answerable']:.3f}, exact={v['selection_set_exact']:.3f}) | "
        f"chal(status={c['status_correct']:.3f}, f1={c['selection_f1_on_answerable']:.3f}, exact={c['selection_set_exact']:.3f})"
    )
sel = summary['selected_checkpoint']
print('\nChallenge slice breakdown for', sel)
print(json.dumps(summary['checkpoints'][sel]['challenge']['slice_breakdown'], ensure_ascii=False, indent=2))